# ONTO-KG Slice 0 — Ontology registry: the shared contract

This notebook builds the **versioned ontology** that every later slice uses as its
single semantic contract (Slice 0, issue #2; epic `docs/epics/EPIC_ONTO_KG.md`).

The reference playbook says: *do not start with LLM extraction prompts — start with
the formal domain model.* We follow that, adapted to synirmos house rules:

- **Internal metamodel first** (`FrozenModel`s, no OWL/SHACL engine): classes with
  `subClassOf` parents, properties with `domain`/`range`, aliases, lifecycle status,
  and a `contextualizing` retrieval tag.
- **Human-readable stable IDs** (`onto:snake_name`) so the ontology stays reviewable.
- **Registry semantics mirror `TopicRegistry`**: collisions fail, aliases attach without
  rewriting history, deprecation supersedes instead of deleting, every mutation bumps
  the working version, and `release()` freezes an immutable fingerprinted artifact.

By the end, you will have a fingerprinted `v1` release of the seed document
ontology and know how it validates extraction triples.

In [ ]:
from datetime import UTC, datetime

from synirmos.processing.ontology import OntologyRegistry, seed_document_ontology

NOW = datetime(2026, 9, 20, 12, 0, tzinfo=UTC)

# The registry starts empty: a working set with version 0 and no content.
empty = OntologyRegistry(version=0, updated_at=NOW)
print(f"empty registry v{empty.version}: {len(empty.classes)} classes")

reg = seed_document_ontology(at=NOW)
print(
    f"seeded registry v{reg.version}: {len(reg.classes)} classes, {len(reg.properties)} properties"
)
for c in reg.classes:
    print(f"  {c.class_id:22} {c.name}")

## Lookup and aliasing: names change, identity doesn't

Concepts are looked up by ID, name, or alias (case-insensitive). Adding an alias
— e.g. the playbook's `StockCorporation` as an alias of a company class — never
rewrites existing references, and claiming another element's alias fails closed.

In [ ]:
print(reg.lookup_class("organization"))
print(reg.lookup_class("onto:organization"))
print(reg.lookup_class("firm"))  # nothing yet

reg, org = reg.add_class_alias("onto:organization", "Firm", at=NOW)
print("aliases:", org.aliases)
print("resolves:", reg.lookup_class("firm").class_id)

# Re-adding your own alias is idempotent; stealing another element's alias fails.
reg, same = reg.add_class_alias("onto:organization", "Firm", at=NOW)
print("idempotent:", same.aliases)
try:
    reg.add_class_alias("onto:person", "firm", at=NOW)
except ValueError as e:
    print("rejected:", e)

## Evolving the model: subclass + a new property

New concepts arrive as genuinely new registrations — never silent merges. A parent
must already exist, and a property's domain/range classes must exist too, so the
model cannot reference what it has not defined.

In [ ]:
from synirmos.processing.ontology import OntologyClass, OntologyProperty

reg, _ = reg.register_class(
    OntologyClass(
        class_id="onto:stock_corporation",
        name="StockCorporation",
        definition="A corporation with issued stock.",
        parent_ids=("onto:organization",),
    ),
    at=NOW,
)
reg, cites = reg.register_property(
    OntologyProperty(
        property_id="onto:cites",
        name="CITES",
        definition="Links a chunk to a document it cites.",
        domains=("onto:chunk",),
        ranges=("onto:document",),
        contextualizing=True,
    ),
    at=NOW,
)
print(f"registered {cites.property_id}; registry v{reg.version}")

# Referencing an undefined class fails closed.
try:
    reg.register_property(
        OntologyProperty(
            property_id="onto:bad",
            name="BAD",
            domains=("onto:chunk",),
            ranges=("onto:missing",),
        ),
        at=NOW,
    )
except ValueError as e:
    print("rejected:", e)

## Validation: the gate every extraction must pass (Slice 3 will call this)

`validate_triple(subject_class, property, object_class)` enforces domain/range with
subclass awareness: a `StockCorporation` may stand wherever an `Organization` is
expected. An empty domain/range list means unconstrained on that side.

In [ ]:
checks = [
    ("onto:chunk", "onto:mentions", "onto:person"),  # chunk names a person
    ("onto:chunk", "onto:mentions", "onto:chunk"),  # chunk is not mentionable
    ("onto:stock_corporation", "onto:part_of", "onto:organization"),  # subclass ok
    ("onto:person", "onto:has_section", "onto:section"),  # people have no sections
    ("onto:chunk", "onto:cites", "onto:document"),  # the new property works
]
for s, p, o in checks:
    print(f"{reg.validate_triple(s, p, o)!s:5}  {s} --{p}--> {o}")

## `compile_schema()`: the closed vocabulary handed to the extractor

Slice 3 will feed exactly this structure to the JERE provider so the LLM can only
propose classes and relations the ontology defines. The `contextualizing` flags are
the playbook's operational semantics: Slice 5's retriever follows only tagged
relations, so retagging changes retrieval with no code change.

In [ ]:
schema = reg.compile_schema()
print(f"schema v{schema['version']}: {len(schema['classes'])} classes")
for p in schema["properties"]:
    mark = "*context*" if p["contextualizing"] else ""
    print(f"  {p['property_id']:22} domains={p['domains']} ranges={p['ranges']} {mark}")

## Immutable releases: fingerprint, extend, supersede

A release freezes the *active* set (deprecated elements are excluded but stay
resolvable for history). Same content → same fingerprint, so releases are
content-addressed; extending the ontology yields a new version that supersedes the
old one. Chunks stamped with an older `ontologyVersion` (Slice 2) become
re-extraction candidates when the ontology grows.

In [ ]:
v1 = reg.release("v1", at=NOW, created_by="notebook-00")
again = reg.release("v1", at=NOW, created_by="notebook-00")
print("v1 fingerprint:", v1.fingerprint)
print("deterministic:", v1.fingerprint == again.fingerprint)

reg, _ = reg.register_class(
    OntologyClass(
        class_id="onto:regulation",
        name="Regulation",
        definition="A rule or directive discussed in text.",
    ),
    at=NOW,
)
v2 = reg.release("v2", at=NOW, created_by="notebook-00", supersedes="v1")
print("v2 supersedes:", v2.supersedes, "| changed:", v2.fingerprint != v1.fingerprint)
print("v2 classes:", len(v2.classes))

## Where this goes next

- **Slice 1 (#3)**: persist these releases as `OntologyClass`/`OntologyProperty` nodes
  in GestaltDB and read the active release back at any horizon.
- **Slice 2 (#4)**: chunk documents; stamp each chunk with `ontologyVersion`.
- **Slice 3 (#5)**: compile this schema into the JERE extractor and route violations
  to staging instead of the graph.